# Módulo 3 — Predicción de Ventas con Prophet
**Prophet (Meta)** sobre Rossmann Store Sales (Kaggle)

> No requiere GPU. CPU es suficiente.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/smartretail360'
DATA_DIR   = f'{DRIVE_ROOT}/data/rossmann'
MODEL_DIR  = f'{DRIVE_ROOT}/models/sales_predictor'

import os
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
!pip install -q prophet plotly scikit-learn joblib
# Para descargar Rossmann:
# !pip install -q kaggle
# !kaggle competitions download -c rossmann-store-sales -p {DATA_DIR} --unzip

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_raw = pd.read_csv(f'{DATA_DIR}/train.csv', parse_dates=['Date'])
print(df_raw.shape)
df_raw.head()

In [ ]:
# EDA — una tienda
STORE_ID = 1
store_df = df_raw[df_raw['Store'] == STORE_ID][['Date', 'Sales']].rename(columns={'Date': 'ds', 'Sales': 'y'})
store_df = store_df.sort_values('ds')
store_df.plot(x='ds', y='y', figsize=(12, 4), title=f'Ventas — Tienda {STORE_ID}')
plt.tight_layout()
plt.show()

In [ ]:
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

cutoff = store_df['ds'].max() - pd.Timedelta(days=30)
train_df = store_df[store_df['ds'] <= cutoff]
test_df  = store_df[store_df['ds'] >  cutoff]

model = Prophet(yearly_seasonality=True, weekly_seasonality=True, interval_width=0.95)
model.fit(train_df)

future   = model.make_future_dataframe(periods=len(test_df))
forecast = model.predict(future)

y_true = test_df['y'].values
y_pred = forecast.tail(len(test_df))['yhat'].values

mae  = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100

print(f'MAE: {mae:.2f} | RMSE: {rmse:.2f} | MAPE: {mape:.2f}%')

In [ ]:
fig = model.plot(forecast)
plt.title(f'Pronóstico Prophet — Tienda {STORE_ID}')
plt.show()

model.plot_components(forecast)
plt.show()

In [ ]:
import joblib
joblib.dump(model, f'{MODEL_DIR}/prophet_model.joblib')
print(f'Prophet guardado en {MODEL_DIR}/prophet_model.joblib')

In [ ]:
# TODO (semana 6): agregar sentimiento como regressor
# sentiment_series = pd.Series(...)  # índice=fecha, valor=score 0-1
# train_df['sentiment'] = train_df['ds'].map(sentiment_series).fillna(0.5)
# model_with_sentiment = Prophet(...)
# model_with_sentiment.add_regressor('sentiment')
# model_with_sentiment.fit(train_df)